In [1]:
import sys
import os

# Add paths to import from long_form_factuality
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)


from vllm_wrapper import VLLMRaterModel
from eval.safe.rate_atomic_fact import check_atomic_fact

/root/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-03 18:46:35.127851142 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


Initializing BM25 Index... (This might take a moment)


/root/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 41 files: 100%|██████████| 41/41 [00:00<00:00, 4341.71it/s]
Dec 03, 2025 6:46:36 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


BM25 Index loaded successfully.


In [2]:
import json
atomic_fact_data = []
# In Jupyter notebooks, use getcwd() instead of __file__
path = os.path.join(os.getcwd(), "data_for_git", "atomic_facts.jsonl")
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        atomic_fact_data.append(json.loads(line))

In [3]:
total_facts = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
print(total_facts)
print(len(atomic_fact_data))
atomic_fact_data[0]["results"]["all_atomic_facts"][0]["atomic_facts"]

124643
3413


['The provided documents do not contain any information about Joeri Adams.']

In [ ]:
llm = VLLMRaterModel()
import threading
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_WORKERS = 1000


# Thread-safe list to collect results
results = []
results_lock = threading.Lock()
error_log = []
error_log_lock = threading.Lock()
pbar_lock = threading.Lock()

def worker(full_dict, pbar):
    try:
        # Extract prompt logic...
        bio_person = full_dict["prompt"].split("Tell me a bio of ")[1]
        
        all_atomic_facts = full_dict["results"]["all_atomic_facts"]
        for sentence_facts in all_atomic_facts:
            for fact_index, fact in enumerate(sentence_facts["atomic_facts"]):
                # 'llm' is captured from outer scope or passed via partial
                rating = check_atomic_fact(fact, bio_person, llm, max_steps=2)[0].answer
                sentence_facts["atomic_facts"][fact_index] = {"fact": fact, "rating": rating}
                with pbar_lock:
                    pbar.update(1)
        
        return ("success", full_dict)
    except Exception as e:
        return ("error", f"Error processing response {full_dict.get('id', '?')}: {e}")

first_n = None

total_facts = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])

tasks = atomic_fact_data[:first_n] if first_n else atomic_fact_data
# 2. Run with Executor
# We create the pbar outside, then pass it into every worker
with tqdm(total=total_facts, desc="Rating Facts") as pbar:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks
        futures = [executor.submit(worker, task, pbar) for task in tasks]
        
        # Wait for completion (so the cell doesn't finish early)
        for future in as_completed(futures):
            # We just iterate to ensure we wait for everyone.
            # Exceptions are caught inside 'worker', so .result() won't throw here.
            _ = future.result()

# 3. Save Logs
with open(os.getcwd() + "/data_for_git/fact_rating_log.txt", "w") as f:
    for error in error_log:
        f.write(error + "\n")

print(f"Processed {len(results)} responses successfully.")

Rating Facts:   0%|          | 0/124643 [00:00<?, ?it/s]

Rating Facts:   1%|          | 1060/124643 [09:17<12:19:03,  2.79it/s]

Error validating output:  The last output contains errors: double brackets, also note that I need to output only [Supported] or [Not Supported]. The format: my final answer should only be either "   		 	,  "final_answer" :  "Supported"  }
Retrying...


Rating Facts:   1%|          | 1169/124643 [10:11<14:33:55,  2.35it/s]

In [23]:
#write results to file with utf8 encoding
with open(os.getcwd() + "/data_for_git/rated_facts.jsonl", "w", encoding="utf-8") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")
